### TRATA BASES DO PROCON E TRANSFOMA EM UMA DELTA ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"     

In [0]:
dbutils.widgets.text("Mes_procon", "", "PROCON")
Mes_procon = dbutils.widgets.get("Mes_procon")

In [0]:
base_procon = f"/Volumes/databox/juridico_comum/arquivos/procon/bases_gerenciais/PROCON_GERENCIAL - {Mes_procon}.xlsx"

df_procon = pd.read_excel(
    base_procon, 
    sheet_name='PROCON',
    header=5
)


In [0]:
# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
df_procon.columns = [clean_col_name(c) for c in df_procon.columns]

In [0]:
pd.set_option('display.max_columns', None)

print(df_procon.columns.to_list())

In [0]:
cols_ids = ['PROCESSO_ID']

for col in cols_ids:
    if col in df_procon.columns:
        df_procon[col] = pd.to_numeric(df_procon[col], errors='coerce').astype('Int64')

In [0]:
cols = pd.Series(df_procon.columns)
for i, col in enumerate(cols[cols.duplicated()]):
    cols[cols.duplicated(keep='first')] = [
        f"{n}_{i+i}" if i != 0 else n
        for i, n in enumerate(cols[cols.duplicated(keep='first')])
    ]
df_procon.columns = cols

In [0]:
# 1. Renomear colunas duplicadas automaticamente (ex: DATA_DA_BAIXA_PROVISORIA_1, _2...)
cols = pd.Series(df_procon.columns)
count = cols.groupby(cols).cumcount()
df_procon.columns = [f"{c}_{i}" if i > 0 else c for c, i in zip(cols, count)]

# 2. Agora que os nomes são únicos, atualizamos a lista de colunas que contêm "DATA"
# Isso garante que pegaremos as originais e as renomeadas (ex: DATA_..._1)
todas_colunas_data = [c for c in df_procon.columns if 'DATA' in c]

# 3. Rodar o seu loop de conversão
for cold in todas_colunas_data:
    df_procon[cold] = pd.to_datetime(df_procon[cold], errors='coerce')

# Visualizar o resultado das colunas de data
print(df_procon[todas_colunas_data].info())

In [0]:
# 1. Identifica todas as colunas que o Pandas entende como 'object' (mistas/strings)
colunas_object = df_procon.select_dtypes(include=['object']).columns

# 2. Converte todas elas explicitamente para string e remove o erro de tipos mistos
for col in colunas_object:
    # O .fillna('') evita que valores nulos virem a string 'nan'
    df_procon[col] = df_procon[col].fillna('').astype(str)

print("Conversão de colunas object concluída.")

# 3. Agora converta para Spark sem medo
df_procon_spark = spark.createDataFrame(df_procon)

In [0]:
print("...")
print("...")
print("...")
print("...")
print("Convertendo para Tabela delta")
df_procon_spark.write.mode('overwrite').saveAsTable('databox.juridico_comum.Silver_Procon_Gerencial')
print("Conversão concluida com sucesso")

In [0]:
%sql
SELECT * FROM databox.juridico_comum.Silver_Procon_Gerencial

### TRATA BASE DO IMOBILIARIO E TRANSFORMA EM UMA DELTA ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"    

In [0]:
dbutils.widgets.text("Mes_imobiliario", "", "IMOBILIARIO")
Mes_imobiliario = dbutils.widgets.get("Mes_imobiliario")

In [0]:
base_imobiliario = f"/Volumes/databox/juridico_comum/arquivos/imobiliário/bases_gerenciais/IMOBILIARIO_GERENCIAL ( CONSOLIDADO) - {Mes_imobiliario}.xlsx"

df_imobiliario = pd.read_excel(
    base_imobiliario, 
    sheet_name='IMOBILIÁRIO',
    header=5
)

In [0]:
# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
df_imobiliario.columns = [clean_col_name(c) for c in df_imobiliario.columns]

In [0]:
pd.set_option('display.max_columns', None)

print(df_imobiliario.columns.to_list())

In [0]:
cols_ids = ['PROCESSO_ID']

for col in cols_ids:
    if col in df_imobiliario.columns:
        df_imobiliario[col] = pd.to_numeric(df_imobiliario[col], errors='coerce').astype('Int64')

In [0]:
cols = pd.Series(df_imobiliario.columns)
for i, col in enumerate(cols[cols.duplicated()]):
    cols[cols.duplicated(keep='first')] = [
        f"{n}_{i+i}" if i != 0 else n
        for i, n in enumerate(cols[cols.duplicated(keep='first')])
    ]
df_imobiliario.columns = cols

In [0]:
# 1. Renomear colunas duplicadas automaticamente (ex: DATA_DA_BAIXA_PROVISORIA_1, _2...)
cols = pd.Series(df_imobiliario.columns)
count = cols.groupby(cols).cumcount()
df_imobiliario.columns = [f"{c}_{i}" if i > 0 else c for c, i in zip(cols, count)]

# 2. Agora que os nomes são únicos, atualizamos a lista de colunas que contêm "DATA"
# Isso garante que pegaremos as originais e as renomeadas (ex: DATA_..._1)
todas_colunas_data = [c for c in df_imobiliario.columns if 'DATA' in c]

# 3. Rodar o seu loop de conversão
for cold in todas_colunas_data:
    df_imobiliario[cold] = pd.to_datetime(df_imobiliario[cold], errors='coerce')

# Visualizar o resultado das colunas de data
print(df_imobiliario[todas_colunas_data].info())

In [0]:
# 1. Identifica todas as colunas que o Pandas entende como 'object' (mistas/strings)
colunas_object = df_imobiliario.select_dtypes(include=['object']).columns

# 2. Converte todas elas explicitamente para string e remove o erro de tipos mistos
for col in colunas_object:
    # O .fillna('') evita que valores nulos virem a string 'nan'
    df_imobiliario[col] = df_imobiliario[col].fillna('').astype(str)

print("Conversão de colunas object concluída.")

# 3. Agora converta para Spark sem medo
df_imobiliario_spark = spark.createDataFrame(df_imobiliario)

In [0]:
print("...")
print("...")
print("...")
print("...")
print("Convertendo para Tabela delta")
df_imobiliario_spark.write.mode('overwrite').saveAsTable('databox.juridico_comum.Silver_Imobiliario_Gerencial')
print("Conversão concluida com sucesso")

In [0]:
%sql
SELECT * FROM databox.juridico_comum.Silver_Imobiliario_Gerencial

### TRATA BASE DO REGULATORIO E TRANSFORMA EM UMA DELTA ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"   

In [0]:
dbutils.widgets.text("Mes_regulatorio", "", "REGULATORIO")
Mes_regulatorio = dbutils.widgets.get("Mes_regulatorio")

In [0]:
base_regulatorio = f"/Volumes/databox/juridico_comum/arquivos/regulatório/bases_gerenciais/REGULATORIO_GERENCIAL (CONSOLIDADO) - {Mes_regulatorio}.xlsx"

df_regulatorio = pd.read_excel(
    base_regulatorio,
    sheet_name='Regulatório',
    header=5
)

In [0]:
# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
df_regulatorio.columns = [clean_col_name(c) for c in df_regulatorio.columns]

In [0]:
pd.set_option('display.max_columns', None)

print(df_regulatorio.columns.to_list())

In [0]:
cols_ids = ['PROCESSO_ID']

for col in cols_ids:
    if col in df_regulatorio.columns:
        df_regulatorio[col] = pd.to_numeric(df_regulatorio[col], errors='coerce').astype('Int64')

In [0]:
cols = pd.Series(df_regulatorio.columns)
for i, col in enumerate(cols[cols.duplicated()]):
    cols[cols.duplicated(keep='first')] = [
        f"{n}_{i+i}" if i != 0 else n
        for i, n in enumerate(cols[cols.duplicated(keep='first')])
    ]
df_regulatorio.columns = cols

In [0]:
# 1. Renomear colunas duplicadas automaticamente (ex: DATA_DA_BAIXA_PROVISORIA_1, _2...)
cols = pd.Series(df_regulatorio.columns)
count = cols.groupby(cols).cumcount()
df_regulatorio.columns = [f"{c}_{i}" if i > 0 else c for c, i in zip(cols, count)]

# 2. Agora que os nomes são únicos, atualizamos a lista de colunas que contêm "DATA"
# Isso garante que pegaremos as originais e as renomeadas (ex: DATA_..._1)
todas_colunas_data = [c for c in df_regulatorio.columns if 'DATA' in c]

# 3. Rodar o seu loop de conversão
for cold in todas_colunas_data:
    df_regulatorio[cold] = pd.to_datetime(df_regulatorio[cold], errors='coerce')

# Visualizar o resultado das colunas de data
print(df_regulatorio[todas_colunas_data].info())

In [0]:
# 1. Identifica todas as colunas que o Pandas entende como 'object' (mistas/strings)
colunas_object = df_regulatorio.select_dtypes(include=['object']).columns

# 2. Converte todas elas explicitamente para string e remove o erro de tipos mistos
for col in colunas_object:
    # O .fillna('') evita que valores nulos virem a string 'nan'
    df_regulatorio[col] = df_regulatorio[col].fillna('').astype(str)

print("Conversão de colunas object concluída.")

# 3. Agora converta para Spark sem medo
df_regulatorio_spark = spark.createDataFrame(df_regulatorio)

In [0]:
print("...")
print("...")
print("...")
print("...")
print("Convertendo para Tabela delta")
df_regulatorio_spark.write.mode('overwrite').saveAsTable('databox.juridico_comum.Silver_Regulatorio_Gerencial')
print("Conversão concluida com sucesso")

In [0]:
%sql
SELECT * FROM databox.juridico_comum.Silver_Regulatorio_Gerencial

### TRATA BASE DO TRABALHISTA GERENCIAL E SOBE EM UMA DELTA ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"   

In [0]:
dbutils.widgets.text("Mes_trabalhista", "", "TRABALHISTA")
Mes_trabalhista = dbutils.widgets.get("Mes_trabalhista")

In [0]:
base_trabalhista_consolidado = f'/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_gerenciais/external/TRABALHISTA_GERENCIAL_(CONSOLIDADOS)-{Mes_trabalhista}.xlsx'

df_trabalhista = pd.read_excel(
    base_trabalhista_consolidado,
    sheet_name='TRABALHISTA',
    header=5
)

In [0]:
# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
df_trabalhista.columns = [clean_col_name(c) for c in df_trabalhista.columns]

In [0]:
pd.set_option('display.max_columns', None)

print(df_trabalhista.columns.to_list())

In [0]:
cols_ids = ['PROCESSO_ID']

for col in cols_ids:
    if col in df_trabalhista.columns:
        df_trabalhista[col] = pd.to_numeric(df_trabalhista[col], errors='coerce').astype('Int64')

In [0]:
cols = pd.Series(df_trabalhista.columns)
for i, col in enumerate(cols[cols.duplicated()]):
    cols[cols.duplicated(keep='first')] = [
        f"{n}_{i+i}" if i != 0 else n
        for i, n in enumerate(cols[cols.duplicated(keep='first')])
    ]
df_trabalhista.columns = cols

In [0]:
# 1. Renomear colunas duplicadas automaticamente (ex: DATA_DA_BAIXA_PROVISORIA_1, _2...)
cols = pd.Series(df_trabalhista.columns)
count = cols.groupby(cols).cumcount()
df_trabalhista.columns = [f"{c}_{i}" if i > 0 else c for c, i in zip(cols, count)]

# 2. Agora que os nomes são únicos, atualizamos a lista de colunas que contêm "DATA"
# Isso garante que pegaremos as originais e as renomeadas (ex: DATA_..._1)
todas_colunas_data = [c for c in df_trabalhista.columns if 'DATA' in c]

# 3. Rodar o seu loop de conversão
for cold in todas_colunas_data:
    df_trabalhista[cold] = pd.to_datetime(df_trabalhista[cold], errors='coerce')

# Visualizar o resultado das colunas de data
print(df_trabalhista[todas_colunas_data].info())

In [0]:
# 1. Identifica todas as colunas que o Pandas entende como 'object' (mistas/strings)
colunas_object = df_trabalhista.select_dtypes(include=['object']).columns

# 2. Converte todas elas explicitamente para string e remove o erro de tipos mistos
for col in colunas_object:
    # O .fillna('') evita que valores nulos virem a string 'nan'
    df_trabalhista[col] = df_trabalhista[col].fillna('').astype(str)

print("Conversão de colunas object concluída.")

# 3. Agora converta para Spark sem medo
df_trabalhista_spark = spark.createDataFrame(df_trabalhista)

In [0]:
print("...")
print("...")
print("...")
print("...")
print("Convertendo para Tabela delta")
df_trabalhista_spark.write.mode('overwrite').saveAsTable('databox.juridico_comum.Silver_Trabalhista_Gerencial')

In [0]:
%sql
SELECT * FROM databox.juridico_comum.Silver_Trabalhista_Gerencial
--WHERE PARTE_CONTRARIA_CPF = '619.123.875-49'
ORDER BY PROCESSO_ID DESC

### CONSOLIDA BASES GERENCIAIS DO CIVIL SEM APLICAÇÃO DE FILTROS ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"  

In [0]:
dbutils.widgets.text("Mes", "", "CIVIL")
Mes = dbutils.widgets.get("Mes")

In [0]:
base_ativos = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Bases_Mensais_Ativo_Encerrado/CIVEL_GERENCIAL_(ATIVOS)-{Mes}.xlsx'
base_encerrados = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Bases_Mensais_Ativo_Encerrado/CIVEL_GERENCIAL_(ENCERRADOS)-{Mes}.xlsx'

df_ativo = pd.read_excel(
    base_ativos,
    sheet_name='CÍVEL',
    header=5
)

df_encerrados = pd.read_excel(
    base_encerrados,
    sheet_name='CÍVEL',
    header=5
)


In [0]:
#Concatena ativos e encerrados sem aplicar as regras do faturametno
df_concatenado_civil = pd.concat([df_ativo, df_encerrados],axis=0)

In [0]:
df_concatenado_civil.shape

In [0]:
# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
df_concatenado_civil.columns = [clean_col_name(c) for c in df_concatenado_civil.columns]

In [0]:
pd.set_option('display.max_columns', None)

# Agora, ao imprimir, todas aparecerão na horizontal
print(df_concatenado_civil.columns.to_list())


In [0]:
# Lista com o nome das suas colunas de ID para transformar em int
cols_id = ['PROCESSO_ID'] 

for col in cols_id:
    if col in df_concatenado_civil.columns:
        # O 'Int64' (maiúsculo) remove os .00 e mantém os vazios como <NA>
        df_concatenado_civil[col] = pd.to_numeric(df_concatenado_civil[col], errors='coerce').astype('Int64')

In [0]:
# Criando um sufixo para nomes duplicados
cols = pd.Series(df_concatenado_civil.columns)
for i, col in enumerate(cols[cols.duplicated()]):
    cols[cols.duplicated(keep='first')] = [
        f"{n}_{i+1}" if i != 0 else n 
        for i, n in enumerate(cols[cols.duplicated(keep='first')])
    ]

df_concatenado_civil.columns = cols

In [0]:
# 1. Renomear colunas duplicadas automaticamente (ex: DATA_DA_BAIXA_PROVISORIA_1, _2...)
cols = pd.Series(df_concatenado_civil.columns)
count = cols.groupby(cols).cumcount()
df_concatenado_civil.columns = [f"{c}_{i}" if i > 0 else c for c, i in zip(cols, count)]

# 2. Agora que os nomes são únicos, atualizamos a lista de colunas que contêm "DATA"
# Isso garante que pegaremos as originais e as renomeadas (ex: DATA_..._1)
todas_colunas_data = [c for c in df_concatenado_civil.columns if 'DATA' in c]

# 3. Rodar o seu loop de conversão
for cold in todas_colunas_data:
    df_concatenado_civil[cold] = pd.to_datetime(df_concatenado_civil[cold], errors='coerce')

# Visualizar o resultado das colunas de data
print(df_concatenado_civil[todas_colunas_data].info())

In [0]:
# 1. Identifica todas as colunas que o Pandas entende como 'object' (mistas/strings)
colunas_object = df_concatenado_civil.select_dtypes(include=['object']).columns

# 2. Converte todas elas explicitamente para string e remove o erro de tipos mistos
for col in colunas_object:
    # O .fillna('') evita que valores nulos virem a string 'nan'
    df_concatenado_civil[col] = df_concatenado_civil[col].fillna('').astype(str)

print("Conversão de colunas object concluída.")

# 3. Agora converta para Spark sem medo
df_civil_spark = spark.createDataFrame(df_concatenado_civil)

In [0]:
print("...")
print("...")
print("...")
print("...")
print("Convertendo para Tabela delta")
df_civil_spark.write.mode('overwrite').saveAsTable('databox.juridico_comum.Silver_Civil_Gerencial')

In [0]:
%sql
SELECT * FROM databox.juridico_comum.Silver_Civil_Gerencial

### Para Consolidado Faturamento Cível###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador" 

In [0]:
print("Começando Leitura do arquivo Excel")

caminho_excel = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Faturamento_honoararios_2023 - 2025/Consolidado Faturamento Honorários Cível Massa 2025.xlsx'

df = pd.read_excel(
    caminho_excel,
    sheet_name='Contrato Novo',
    header=1
)

print("Leitura realizada com sucesso")

In [0]:
# ---------------------------------------------------------
# 1. RENOMEAR COLUNAS (Padronização Snake Case)
# ---------------------------------------------------------
# Mapeamento manual para colunas críticas ou muito longas
rename_map = {
    'MÊS/ANO': 'MES_ANO',
    'PROCESSO - ID': 'PROCESSO_ID',
    'SUB-ÁREA DO DIREITO': 'SUB_AREA_DIREITO',
    'ASSUNTO (CÍVEL) - PRINCIPAL': 'ASSUNTO_CIVEL_PRINCIPAL',
    'VALOR QUE O ESCRITÓRIO CONSIDERA QUE DEVE SER PAGO(APONTAR VALOR APENAS PARA OS CASOS EM QUE O ESCRITORIO NÃO CONCORDA COM O VALOR A SER PAGO, DEVENDO SER PREENCHIDO A COLUNA BO TAMBÉM)': 'VALOR_PLEITEADO_ESCRITORIO',
    'QUANT. PARCELA MENSALIDADE': 'QTD_PARCELA_MENSALIDADE',
    'MENSALIDADE JEC (10 meses)': 'MENSALIDADE_JEC_10M',
    'MENSALIDADE VARA COMUM (18 meses)': 'MENSALIDADE_VARA_COMUM_18M',
    'Chave (ID + Mês Pagamento)': 'CHAVE_ID_MES_PAGTO',
    'Chave (ID + Acordo)': 'CHAVE_ID_ACORDO',
    'Chave (ID + Improcedência)': 'CHAVE_ID_IMPROCEDENCIA',
    'Chave (ID + STATUS)': 'CHAVE_ID_STATUS'
}

# Aplica o rename manual
pdf = df.rename(columns=rename_map)

# Função para limpar o restante automaticamente (remove acentos, espaços, uppercase)
def clean_col_name(col):
    col = col.lower()
    col = col.replace('ç', 'c').replace('ã', 'a').replace('õ', 'o').replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
    col = re.sub(r'[^a-z0-9_]', '_', col) # Substitui tudo que não é letra/num por _
    col = re.sub(r'_+', '_', col)         # Remove underlines duplicados (__)
    return col.strip('_')

pdf.columns = [clean_col_name(c) for c in pdf.columns]

# ---------------------------------------------------------
# 2. DEFINIR GRUPOS DE COLUNAS PARA TRATAMENTO
# ---------------------------------------------------------
# Identifiquei estas colunas financeiras que precisam ser Float
cols_financeiras = [
    'VALOR_TOTAL_A_PAGAR', 'GLOSA', 'BONUS_ACORDO', 'BONUS_IMPROCEDENCIA',
    'MENSALIDADE_5_PARCELAS', 'MENSALIDADE_JEC_10M', 'MENSALIDADE_VARA_COMUM_18M',
    'VALOR_PLEITEADO_ESCRITORIO'
]

# Colunas de Data
cols_datas = ['DATA_REGISTRO', 'DATA_CONFIRMACAO', 'DATA_DE_REGISTRO_DO_ENCERRAMENTO']

# ---------------------------------------------------------
# 3. TRATAMENTO DE DADOS (LIMPEZA)
# ---------------------------------------------------------

# A) Tratamento de Floats (Financeiro)
for col in cols_financeiras:
    if col in pdf.columns:
        # Converte para string primeiro para limpar "R$" e ","
        pdf[col] = pdf[col].astype(str).str.replace('R$', '', regex=False)
        pdf[col] = pdf[col].str.replace('.', '', regex=False) # Tira ponto de milhar
        pdf[col] = pdf[col].str.replace(',', '.', regex=False) # Troca vírgula por ponto decimal
        # Força numérico, erros viram NaN, depois preenche NaN com 0.0
        pdf[col] = pd.to_numeric(pdf[col], errors='coerce').fillna(0.0)

# B) Tratamento de Datas
for col in cols_datas:
    if col in pdf.columns:
        pdf[col] = pd.to_datetime(pdf[col], errors='coerce', dayfirst=True)

# C) Tratamento Geral de Strings (Isso resolve seu erro de PySparkTypeError)
# Pega todas as colunas que sobraram como 'object'
cols_object = pdf.select_dtypes(include=['object']).columns

for col in cols_object:
    # Converte para string e trata nulos
    pdf[col] = pdf[col].astype(str).replace({'nan': None, 'NaT': None, '': None})

In [0]:
print("2. Convertendo para o formato do Spark...")

br_Consolidado_Honararios = spark.createDataFrame(pdf)

# Célula 4 corrigida
br_Consolidado_Honararios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databox.juridico_comum.br_Consolidado_Honorarios") 

print("Sucesso!")

In [0]:
%sql
select * from databox.juridico_comum.br_Consolidado_Honorarios
where processo_id in ('920460', '946967', '549574', '1320705', '1324024', '1316112')

### Consolidado Calculos e Pericias###

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
import numpy as np
import re

In [0]:
base_consolidado_pericias = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Arquivo_Consolidado/Consolidado Faturamento - Calculos e Perícias 2023 e 2025.xlsx'

df_pericias = pd.read_excel(
    base_consolidado_pericias,
    sheet_name='PERÍCIAS',
    header=1,
    engine='openpyxl'
)

df_calculos = pd.read_excel(
    base_consolidado_pericias,
    sheet_name='CÁLCULOS',
    header=1,
    engine='openpyxl'
)

In [0]:
pdf_pericias = df_pericias

# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
pdf_pericias.columns = [clean_col_name(c) for c in pdf_pericias.columns]

In [0]:
pdf_pericias.head()

In [0]:
# Colunas numéricas (Floats)
cols_financeiras = [
    'PRECO_INICIAL', 
    'PERCENTUAL_A_PAGAR_DA_PERICIA', 
    'PRECO_FINAL'
]

# Colunas de data (Datetime)
cols_datas = [
    'DATA_DE_CONFIRMACAO', 
    'PRAZO_SLA', 
    'COMPETENCIA'
]

In [0]:
for col in cols_financeiras:
    if col in pdf_pericias.columns:
        # Converte para string primeiro para limpar "R$" e ","
        pdf_pericias[col] = pdf_pericias[col].astype(str).str.replace('R$', '', regex=False)
        pdf_pericias[col] = pdf_pericias[col].str.replace('.', '', regex=False) # Tira ponto de milhar
        pdf_pericias[col] = pdf_pericias[col].str.replace(',', '.', regex=False) # Troca vírgula por ponto decimal
        # Força numérico, erros viram NaN, depois preenche NaN com 0.0
        pdf_pericias[col] = pd.to_numeric(pdf_pericias[col], errors='coerce').fillna(0.0)

# B) Tratamento de Datas
for col in cols_datas:
    if col in pdf_pericias.columns:
        pdf_pericias[col] = pd.to_datetime(pdf_pericias[col], errors='coerce', dayfirst=True)

# C) Tratamento Geral de Strings (Isso resolve seu erro de PySparkTypeError)
# Pega todas as colunas que sobraram como 'object'
cols_object = pdf_pericias.select_dtypes(include=['object']).columns

for col in cols_object:
    # Converte para string e trata nulos
    pdf_pericias[col] = pdf_pericias[col].astype(str).replace({'nan': None, 'NaT': None, '': None})

In [0]:
# Lista com o nome das suas colunas de ID
cols_id = ['PROCESSO_ID', 'TAREFA_ID', 'FICHAS_DE_PROCESSO_ID'] 

for col in cols_id:
    if col in pdf_pericias.columns:
        # O 'Int64' (maiúsculo) remove os .00 e mantém os vazios como <NA>
        pdf_pericias[col] = pd.to_numeric(pdf_pericias[col], errors='coerce').astype('Int64')

In [0]:
pdf_pericias.head()

In [0]:
# 1. Reseta o índice e remove colunas vazias
pdf_pericias = pdf_pericias.reset_index(drop=True)
pdf_pericias = pdf_pericias.loc[:, pdf_pericias.columns != '']

# --- NOVA PARTE: Verificação de Segurança ---
print(f"Dimensões do Pandas DF: {pdf_pericias.shape} (linhas, colunas)")
print(f"Nomes das colunas: {pdf_pericias.columns.tolist()}")

# Só prossegue com a conversão se o DataFrame não estiver vazio e tiver colunas
if not pdf_pericias.empty and len(pdf_pericias.columns) > 0:
    
    # Converte de volta para Spark
    br_Consolidado_Pericias = spark.createDataFrame(pdf_pericias)
    
    # Salva no Unity Catalog / Delta
    br_Consolidado_Pericias.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("databox.juridico_comum.br_Consolidado_Pericias")
        
    print("Sucesso! Tabela salva.")
    
else:
    print("🚨 ERRO: O DataFrame Pandas ficou totalmente vazio após a limpeza. Nenhuma tabela foi salva.")

In [0]:
%sql
select * from databox.juridico_comum.br_consolidado_pericias
WHERE ANO = 2026


### Consolidado Calculos ###

In [0]:
df_calculos.columns

In [0]:
pdf_calculos = df_calculos

# Função corrigida
def clean_col_name(col):
    col = str(col).upper() # Garante que é string e passa para maiúsculo
    col = col.replace('Ç', 'C').replace('Á', 'A').replace('À', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    col = col.replace('Ã', 'A').replace('Õ', 'O').replace('Â', 'A').replace('Ê', 'E').replace('Ô', 'O') # Adicionei mais alguns acentos comuns
    
    # CORREÇÃO: Permitindo letras maiúsculas (A-Z)
    col = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
    
    col = re.sub(r'_+', '_', col)  # Remove underlines duplicados
    return col.strip('_')

# Aplica a função
pdf_calculos.columns = [clean_col_name(c) for c in pdf_calculos.columns]

In [0]:
cols_financeiras_1 = ['PRECO_INICIAL', 'VALOR_PENALIDADE_FORA_SLA', 'PRECO_FINAL'] # Colunas que virarão float
cols_datas_1 = ['DATA_DE_CONFIRMACAO', 'COMPETENCIA', 'PRAZO_SLA', 'DATA_DE_CONFIRMACAO'] # Colunas que virarão datetime

In [0]:
for col in cols_financeiras_1:
    if col in pdf_calculos.columns:
        # Converte para string primeiro para limpar "R$" e ","
        pdf_calculos[col] = pdf_calculos[col].astype(str).str.replace('R$', '', regex=False)
        pdf_calculos[col] = pdf_calculos[col].str.replace('.', '', regex=False) # Tira ponto de milhar
        pdf_calculos[col] = pdf_calculos[col].str.replace(',', '.', regex=False) # Troca vírgula por ponto decimal
        # Força numérico, erros viram NaN, depois preenche NaN com 0.0
        pdf_calculos[col] = pd.to_numeric(pdf_calculos[col], errors='coerce').fillna(0.0)

# B) Tratamento de Datas
for col in cols_datas_1:
    if col in pdf_calculos.columns:
        pdf_calculos[col] = pd.to_datetime(pdf_calculos[col], errors='coerce', dayfirst=True)

# C) Tratamento Geral de Strings (Isso resolve seu erro de PySparkTypeError)
# Pega todas as colunas que sobraram como 'object'
cols_object = pdf_calculos.select_dtypes(include=['object']).columns

for col in cols_object:
    # Converte para string e trata nulos
    pdf_calculos[col] = pdf_calculos[col].astype(str).replace({'nan': None, 'NaT': None, '': None})

In [0]:
# Lista com o nome das suas colunas de ID
cols_id = ['PROCESSO_ID', 'TAREFA_ID', 'FICHAS_DE_PROCESSO_ID', 'ID'] 

for col in cols_id:
    if col in pdf_calculos.columns:
        # O 'Int64' (maiúsculo) remove os .00 e mantém os vazios como <NA>
        pdf_calculos[col] = pd.to_numeric(pdf_calculos[col], errors='coerce').astype('Int64')

In [0]:
pdf_calculos.head()

In [0]:
print("2. Convertendo para o formato do Spark...")

br_Consolidado_Calculos = spark.createDataFrame(pdf_calculos)

# Célula 4 corrigida
br_Consolidado_Calculos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databox.juridico_comum.br_Consolidado_Calculos") 

print("Sucesso!")

In [0]:
%sql
select * from databox.juridico_comum.br_consolidado_calculos
WHERE ANO = 2026

## Base Consolidada INSS Decaido ##

In [0]:
base_inss = f'/Volumes/databox/juridico_comum/arquivos/Normalizador_de_bases/INSS/Base Consolidada INSS Decaído 2023 e 2025.xlsx'

df_inss = pd.read_excel(
    base_inss,
    sheet_name="Base INSS",
    header=2,
    engine="openpyxl"
)

In [0]:
import unicodedata
import re

def clean_column_names(df):
    new_columns = []
    for col in df_inss.columns:
        # 1. Remove acentos e converte para minúsculas
        nksel = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('ascii').lower()
        # 2. Substitui qualquer caractere que não seja letra ou número por underscore
        clean_name = re.sub(r'[^a-zA-Z0-9]+', '_', nksel)
        # 3. Remove underscores duplicados ou no final do nome
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        new_columns.append(clean_name)
    
    df_inss.columns = new_columns
    return df_inss

# Aplicando no seu DataFrame
df_inss = clean_column_names(df_inss)

# Converte todos os nomes de colunas para maiúsculo
df_inss.columns = [c.upper() for c in df_inss.columns]

# Visualizar o resultado
print(df_inss.columns)

In [0]:
print("2. Convertendo para o formato do Spark...")

br_Consolidado_INSS = spark.createDataFrame(df_inss)

# Célula 4 corrigida
br_Consolidado_INSS.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databox.juridico_comum.br_Consolidado_INSS") 

print("Sucesso!")

In [0]:
%sql
select * from databox.juridico_comum.br_consolidado_INSS

### Civil consolidado ###

In [0]:
Civil_conso_mensal = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Gerencial_Consolidada_Final/BASE_GERENCIAL - 20260121_Consolidada.xlsx'

df_Civil_conso_mensal = pd.read_excel(
    Civil_conso_mensal,
    sheet_name="CÍVEL",
    header=5
    )

In [0]:
df_Civil_conso_mensal.head()

In [0]:
import pandas as pd
import re
import unicodedata

def tratar_dataframe(df_input):
    df = df_input.copy()

    # --- 1. Limpeza de Nomes de Colunas ---
    novas_colunas = []
    for col in df.columns:
        col_str = unicodedata.normalize('NFKD', str(col)).encode('ASCII', 'ignore').decode('utf-8')
        col_str = col_str.replace(' ', '_').replace('-', '_')
        col_str = re.sub(r'[^a-zA-Z0-9_]', '', col_str)
        col_str = re.sub(r'_+', '_', col_str).strip('_')
        novas_colunas.append(col_str.upper())
    
    df.columns = novas_colunas

    # Lista para rastrear quais colunas já tratamos
    colunas_ja_tratadas = []

    # --- 2. Conversão de PROCESSO_ID para Int ---
    if 'PROCESSO_ID' in df.columns:
        df['PROCESSO_ID'] = pd.to_numeric(df['PROCESSO_ID'], errors='coerce').astype('Int64')
        colunas_ja_tratadas.append('PROCESSO_ID')

    # --- 3. Conversão de Colunas de DATA para Datetime ---
    for col in df.columns:
        if 'DATA' in col:
            df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
            colunas_ja_tratadas.append(col)

    # --- 4. Tudo que sobrou vira String (Texto) ---
    for col in df.columns:
        if col not in colunas_ja_tratadas:
            # Converte para string
            df[col] = df[col].astype(str)
            
            # Limpeza extra opcional: 'astype(str)' transforma valores vazios (NaN) 
            # na palavra escrita "nan". A linha abaixo substitui "nan" por vazio real.
            df[col] = df[col].replace('nan', '')

    return df

# ---------------------------------------------------------

# Aplicação
df_Civil_conso_mensal = tratar_dataframe(df_Civil_conso_mensal)

# Verificação dos tipos
print(df_Civil_conso_mensal.dtypes)
print("-" * 30)
print(df_Civil_conso_mensal.head())

In [0]:
print("2. Convertendo para o formato do Spark...")

br_Civil_conso_mensal = spark.createDataFrame(df_Civil_conso_mensal)

# Célula 4 corrigida
br_Civil_conso_mensal.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databox.juridico_comum.br_Civil_conso_mensal") 

print("Sucesso!")

In [0]:
%sql
SELECT * FROM databox.juridico_comum.br_Civil_conso_mensal
WHERE PROCESSO_ID = 1307551
